# YCrCb threshold ile ateş tespiti

Y > 200 (parlaklık), Cr > Cb (kırmızı > yeşil)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

PATH = "frame_0161.png"

# threshold ayarları
Y_MIN = 200      # parlaklık
CR_CB_DIFF = 0   # Cr - Cb > bu değer (0 => Cr > Cb)


In [ ]:
bgr = cv2.imread(PATH)
assert bgr is not None, f"bulunamadı: {PATH}"
ycrcb = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)
Y, Cr, Cb = cv2.split(ycrcb)

def segment(y_min=Y_MIN, diff=CR_CB_DIFF):
    mask = (Y > y_min) & (Cr.astype(int) - Cb.astype(int) > diff)
    mask = mask.astype(np.uint8) * 255
    fire = cv2.bitwise_and(bgr, bgr, mask=mask)
    return mask, fire

def show(y_min=Y_MIN, diff=CR_CB_DIFF):
    mask, fire = segment(y_min, diff)
    n = int((mask > 0).sum())
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); axs[0].set_title("orijinal")
    axs[1].imshow(mask, cmap="gray"); axs[1].set_title(f"maske ({n} piksel, %{100*n/mask.size:.2f})")
    axs[2].imshow(cv2.cvtColor(fire, cv2.COLOR_BGR2RGB)); axs[2].set_title(f"Y>{y_min}, Cr-Cb>{diff}")
    for a in axs: a.axis("off")
    plt.tight_layout(); plt.show()

show()


## Kanal dağılımı (ayar yapmak için)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3))
for ax, ch, name in zip(axs, (Y, Cr, Cb), ("Y", "Cr", "Cb")):
    ax.hist(ch.ravel(), bins=256, range=(0, 255)); ax.set_title(name)
axs[0].axvline(Y_MIN, color="r", linestyle="--", label=f"Y_MIN={Y_MIN}")
axs[0].legend()
plt.tight_layout(); plt.show()


## Cr - Cb farkı dağılımı

In [ ]:
diff_map = Cr.astype(int) - Cb.astype(int)
plt.figure(figsize=(12, 4))
plt.hist(diff_map.ravel(), bins=256, range=(-255, 255))
plt.axvline(0, color="r", linestyle="--", label="Cr=Cb")
plt.xlabel("Cr - Cb")
plt.title("Cr - Cb farkı dağılımı")
plt.legend()
plt.show()


## Slider ile ayar

In [ ]:
from ipywidgets import interact, IntSlider

interact(show,
         y_min=IntSlider(value=Y_MIN, min=0, max=255, continuous_update=False, description="Y min"),
         diff=IntSlider(value=CR_CB_DIFF, min=-100, max=100, continuous_update=False, description="Cr-Cb diff"));


## Kaydet

In [ ]:
mask, fire = segment()
cv2.imwrite("fire_mask.png", mask)
cv2.imwrite("fire_extracted.png", fire)
print(f"Kaydedildi: fire_mask.png, fire_extracted.png")
print(f"Seçilen piksel sayısı: {(mask > 0).sum()}")
